#00. Imports & Config

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import json

In [0]:
dbutils.widgets.text("ORIGIN_TABLE", "samples.bakehouse.sales_transactions")
dbutils.widgets.text("ORIGIN_PERIOD_COLUMN", "dateTime")
dbutils.widgets.text("ORIGIN_KEY_PARTS", json.dumps(["product", "paymentMethod"]))
dbutils.widgets.text("ORIGIN_Y_COL", "quantity")

dbutils.widgets.text("HIST_PERIOD_COL", "dt_period")
dbutils.widgets.text("HIST_KEY_PARTS", json.dumps(["product", "paymentMethod"]))
dbutils.widgets.text("HIST_Y_COL", "qt_units")

dbutils.widgets.text("MACRO_FORECAST_TABLE", "workspace.forecasting.macro_forecast")
dbutils.widgets.text("MACRO_FORECAST_PERIOD_COLUMN", "dt_period")
dbutils.widgets.text("MACRO_FORECAST_KEY_COLUMN", "cd_key")
dbutils.widgets.text("MACRO_KEY_SEPARATOR", ";")
dbutils.widgets.text("MACRO_KEY_PARTS", json.dumps(["product"]))
dbutils.widgets.text("MACRO_FORECAST_Y_PRED_COLUMN", "y_pred")

dbutils.widgets.text("MICRO_FORECAST_TABLE", "workspace.forecasting.linear_regression_forecast")
dbutils.widgets.text("MICRO_FORECAST_PERIOD_COLUMN", "dt_period")
dbutils.widgets.text("MICRO_FORECAST_KEY_COLUMN", "cd_key")
dbutils.widgets.text("MICRO_KEY_SEPARATOR", ";")
dbutils.widgets.text("MICRO_KEY_PARTS", json.dumps(["product", "paymentMethod"]))
dbutils.widgets.text("MICRO_FORECAST_Y_PRED_COLUMN", "qt_units")

dbutils.widgets.text("TARGET_TABLE", "workspace.forecasting.bakehouse_forecasting")

dbutils.widgets.text("CAP_LAUNCH", "0.25")
dbutils.widgets.text("CAP_MATURE", "0.60")

dbutils.widgets.text("PRIOR_SMOOTH_FRAC", "0.02")

dbutils.widgets.text("BETA_LAUNCH_MIN", "2.0")
dbutils.widgets.text("BETA_LAUNCH_MAX", "10.0")
dbutils.widgets.text("BETA_MATURE_MIN", "1.0")
dbutils.widgets.text("BETA_MATURE_MAX", "4.0")

dbutils.widgets.text("USE_SQRT_PRODUCT_FOR_LAUNCH_RELIABILITY", "true")

In [0]:
def w_str(name: str) -> str:
    return dbutils.widgets.get(name).strip()

def w_json(name: str):
    return json.loads(w_str(name))

def w_int(name: str) -> int:
    return int(w_str(name))

def w_float(name: str) -> float:
    return float(w_str(name))

def w_bool(name: str) -> bool:
    v = w_str(name).lower()
    if v in ("true", "1", "yes", "y", "t"):  return True
    if v in ("false", "0", "no", "n", "f"): return False
    raise ValueError(f"Invalid boolean for {name}: {v}")

In [0]:
ORIGIN_TABLE = w_str("ORIGIN_TABLE")
ORIGIN_PERIOD_COLUMN = w_str("ORIGIN_PERIOD_COLUMN")
ORIGIN_KEY_PARTS = w_json("ORIGIN_KEY_PARTS")
ORIGIN_Y_COL = w_str("ORIGIN_Y_COL")

HIST_PERIOD_COL = w_str("HIST_PERIOD_COL")
HIST_KEY_PARTS = w_json("HIST_KEY_PARTS")
HIST_Y_COL = w_str("HIST_Y_COL")

MACRO_FORECAST_TABLE = w_str("MACRO_FORECAST_TABLE")
MACRO_FORECAST_PERIOD_COLUMN = w_str("MACRO_FORECAST_PERIOD_COLUMN")
MACRO_FORECAST_KEY_COLUMN = w_str("MACRO_FORECAST_KEY_COLUMN")
MACRO_KEY_SEPARATOR = w_str("MACRO_KEY_SEPARATOR")
MACRO_KEY_PARTS = w_json("MACRO_KEY_PARTS")
MACRO_FORECAST_Y_PRED_COLUMN = w_str("MACRO_FORECAST_Y_PRED_COLUMN")

MICRO_FORECAST_TABLE = w_str("MICRO_FORECAST_TABLE")
MICRO_FORECAST_PERIOD_COLUMN = w_str("MICRO_FORECAST_PERIOD_COLUMN")
MICRO_FORECAST_KEY_COLUMN = w_str("MICRO_FORECAST_KEY_COLUMN")
MICRO_KEY_SEPARATOR = w_str("MICRO_KEY_SEPARATOR")
MICRO_KEY_PARTS = w_json("MICRO_KEY_PARTS")
MICRO_FORECAST_Y_PRED_COLUMN = w_str("MICRO_FORECAST_Y_PRED_COLUMN")

TARGET_TABLE = w_str("TARGET_TABLE")

CAP_LAUNCH = w_float("CAP_LAUNCH")
CAP_MATURE = w_float("CAP_MATURE")

PRIOR_SMOOTH_FRAC = w_float("PRIOR_SMOOTH_FRAC")

BETA_LAUNCH_MIN = w_float("BETA_LAUNCH_MIN")
BETA_LAUNCH_MAX = w_float("BETA_LAUNCH_MAX")
BETA_MATURE_MIN = w_float("BETA_MATURE_MIN")
BETA_MATURE_MAX = w_float("BETA_MATURE_MAX")

USE_SQRT_PRODUCT_FOR_LAUNCH_RELIABILITY = w_bool("USE_SQRT_PRODUCT_FOR_LAUNCH_RELIABILITY")

if CAP_LAUNCH < 0 or CAP_MATURE < 0 or CAP_LAUNCH > 1 or CAP_MATURE > 1:
    raise ValueError("CAP_LAUNCH and CAP_MATURE must be between 0 and 1.")
if CAP_LAUNCH >= CAP_MATURE:
    raise ValueError("CAP_LAUNCH must be < CAP_MATURE.")
if not ORIGIN_KEY_PARTS or not isinstance(ORIGIN_KEY_PARTS, list):
    raise ValueError("ORIGIN_KEY_PARTS must be a non-empty JSON list.")

In [0]:
def build_key_split_columns(key_column, separator, key_parts):
    columns = []
    
    for idx, part_name in enumerate(key_parts):
        columns.append(
            f"split({key_column}, '{separator}')[{idx}] AS {part_name}"
        )
    
    return ",\n    ".join(columns)

In [0]:
cutoff = spark.sql(f"SELECT MAX(DATE({ORIGIN_PERIOD_COLUMN})) AS cutoff FROM {ORIGIN_TABLE}").first()["cutoff"]
cutoff_lit = F.lit(cutoff)
print("Cutoff:", cutoff)

#01. Macro Definition

In [0]:
split_columns_sql = build_key_split_columns(
    MACRO_FORECAST_KEY_COLUMN,
    MACRO_KEY_SEPARATOR,
    MACRO_KEY_PARTS
)

query = f"""
SELECT
    {MACRO_FORECAST_PERIOD_COLUMN},
    {split_columns_sql},
    {MACRO_FORECAST_Y_PRED_COLUMN}
FROM {MACRO_FORECAST_TABLE}
WHERE {MACRO_FORECAST_PERIOD_COLUMN} >
      (SELECT ADD_MONTHS(MAX(DATE({ORIGIN_PERIOD_COLUMN})), -1) FROM {ORIGIN_TABLE})
"""

macro_df = spark.sql(query)

group_cols = [MACRO_FORECAST_PERIOD_COLUMN] + MACRO_KEY_PARTS

macro = (
    macro_df
    .select(
        *group_cols,
        F.col(MACRO_FORECAST_Y_PRED_COLUMN).cast("double").alias("macro_qt_units")
    )
    .groupBy(*group_cols)
    .agg(F.sum("macro_qt_units").alias("macro_qt_units"))
)

macro_keys = macro.select(*group_cols).distinct()

#02. Micro Definition

In [0]:
split_columns_sql = build_key_split_columns(
    MICRO_FORECAST_KEY_COLUMN,
    MICRO_KEY_SEPARATOR,
    MICRO_KEY_PARTS
)

query = f"""
SELECT
{MICRO_FORECAST_PERIOD_COLUMN},
{split_columns_sql},
{MICRO_FORECAST_Y_PRED_COLUMN}
FROM
{MICRO_FORECAST_TABLE}
WHERE 
{MICRO_FORECAST_PERIOD_COLUMN} > (SELECT ADD_MONTHS(MAX(DATE({ORIGIN_PERIOD_COLUMN})), -1) FROM {ORIGIN_TABLE})
AND fl_type = 'Linear Regression'
"""

micro_df = spark.sql(query)

period_col = MICRO_FORECAST_PERIOD_COLUMN          # "dt_period"
y_col = MICRO_FORECAST_Y_PRED_COLUMN               # "qt_units"
group_cols = [period_col] + MICRO_KEY_PARTS

micro = (
    micro_df
    .select(
        *group_cols,
        F.col(y_col).cast("double").alias("micro_qt_units")
    )
    .groupBy(*group_cols)
    .agg(F.sum("micro_qt_units").alias("micro_qt_units"))
)

#03. Hist Definiton

In [0]:
def build_origin_agg_df(
    spark,
    origin_table: str,
    period_col: str,
    key_parts: list[str],
    y_col: str,
    cutoff_ts: str,
):
    df = spark.table(origin_table)

    df = (
        df
        .filter(F.col(period_col) <= F.to_timestamp(F.lit(cutoff_ts)))
        .withColumn("dt_period", F.to_date(F.col(period_col)))
        .select("dt_period", *key_parts, F.col(y_col).alias("qt_units_raw"))
    )

    group_cols = ["dt_period"] + key_parts

    return (
        df.groupBy(*group_cols)
          .agg(F.sum("qt_units_raw").alias("qt_units"))
    )

hist_df = build_origin_agg_df(
    spark,
    ORIGIN_TABLE,
    ORIGIN_PERIOD_COLUMN,
    ORIGIN_KEY_PARTS,
    ORIGIN_Y_COL,
    cutoff
)

hist_df = hist_df.select(
    HIST_PERIOD_COL,
    *HIST_KEY_PARTS,
    F.col(HIST_Y_COL).cast("double").alias("qt_unidade")
)

#04. SKU metadata + priors

In [0]:

period_col     = HIST_PERIOD_COL
market_cols    = MACRO_KEY_PARTS
sku_cols       = MICRO_KEY_PARTS
hist_units_col = "hist_qt_units"

hist_df2 = (
    hist_df
    .select(
        F.col(period_col).alias(period_col),
        *[F.col(c) for c in sku_cols],
        F.col("qt_unidade").cast("double").alias(hist_units_col)
    )
)

w_mkt = Window.partitionBy(*market_cols)

sku_stats = (
    hist_df2
    .filter(F.col(period_col) <= cutoff_lit)
    .groupBy(*sku_cols)
    .agg(
        F.min(F.when(F.col(hist_units_col) > 0, F.col(period_col))).alias("first_sale_dt"),
        F.countDistinct(F.when(F.col(hist_units_col) > 0, F.col(period_col))).alias("n_periods_with_sale"),
        F.sum(
            F.when(F.col(period_col) > F.add_months(cutoff_lit, -12), F.col(hist_units_col)).otherwise(F.lit(0.0))
        ).alias("sum_12m"),
        F.sum(
            F.when(F.col(period_col) > F.add_months(cutoff_lit, -6), F.col(hist_units_col)).otherwise(F.lit(0.0))
        ).alias("sum_6m"),
        F.sum(
            F.when(F.col(period_col) > F.add_months(cutoff_lit, -24), F.col(hist_units_col)).otherwise(F.lit(0.0))
        ).alias("sum_24m"),
    )
    .withColumn("first_sale_dt", F.coalesce(F.col("first_sale_dt"), cutoff_lit))
    .withColumn("age_months_cutoff", F.floor(F.months_between(cutoff_lit, F.col("first_sale_dt"))) + F.lit(1))
    .withColumn("is_launch", F.when(F.col("age_months_cutoff") <= 24, F.lit(1)).otherwise(F.lit(0)))
    .withColumn("base_units", F.when(F.col("is_launch") == 1, F.col("sum_6m")).otherwise(F.col("sum_12m")))
)

sku_stats = (
    sku_stats
    .withColumn("n_skus_mkt", F.count(F.lit(1)).over(w_mkt))
    .withColumn("sum_base_mkt", F.sum("base_units").over(w_mkt))
    .withColumn(
        "avg_base_mkt",
        F.when(F.col("n_skus_mkt") > 0, F.col("sum_base_mkt") / F.col("n_skus_mkt")).otherwise(F.lit(0.0))
    )
    .withColumn("alpha", F.col("avg_base_mkt") * F.lit(PRIOR_SMOOTH_FRAC))
    .withColumn(
        "prior_share",
        F.when(
            (F.col("sum_base_mkt") + F.col("alpha") * F.col("n_skus_mkt")) > 0,
            (F.col("base_units") + F.col("alpha")) / (F.col("sum_base_mkt") + F.col("alpha") * F.col("n_skus_mkt"))
        ).otherwise(F.lit(None))
    )
)

priors = sku_stats.select(
    *sku_cols,
    "first_sale_dt",
    "n_periods_with_sale",
    "is_launch",
    "sum_24m",
    "prior_share"
)

#05. SKU universe for allocation

In [0]:
micro_skus = micro.select(*sku_cols).distinct()

active_skus = (
    priors
    .filter(F.col("sum_24m") > 0)
    .select(*sku_cols)
    .distinct()
)

sku_universe = micro_skus.unionByName(active_skus).distinct()

sku_meta = (
    sku_universe
    .join(priors, on=sku_cols, how="left")
    .withColumn("first_sale_dt", F.coalesce(F.col("first_sale_dt"), cutoff_lit))
    .withColumn("n_periods_with_sale", F.coalesce(F.col("n_periods_with_sale"), F.lit(0)))
    .withColumn("is_launch", F.coalesce(F.col("is_launch"), F.lit(1)))
)

In [0]:
skeleton = (
    macro_keys
    .join(
        sku_meta.select(*market_cols, *[c for c in sku_cols if c not in market_cols],
                        "first_sale_dt", "n_periods_with_sale", "is_launch", "prior_share"),
        on=market_cols,
        how="inner"
    )
)

join_micro_cols = [period_col] + sku_cols

base = (
    skeleton
    .join(micro, on=join_micro_cols, how="left")
    .withColumn("micro_qt_units", F.coalesce(F.col("micro_qt_units"), F.lit(0.0)))
)

#06. Robust weights per market-period

In [0]:
w_group = Window.partitionBy(period_col, *market_cols)

base = (
    base
    .withColumn("cnt_sku", F.count(F.lit(1)).over(w_group))
    .withColumn("prior_share", F.coalesce(F.col("prior_share"), F.lit(1.0) / F.col("cnt_sku")))
    .withColumn("micro_pos", F.when(F.col("micro_qt_units") > 0, F.col("micro_qt_units")).otherwise(F.lit(0.0)))
    .withColumn("sum_micro", F.sum("micro_pos").over(w_group))
    .withColumn(
        "micro_share",
        F.when(F.col("sum_micro") > 0, F.col("micro_pos") / F.col("sum_micro"))
         .otherwise(F.col("prior_share"))
    )
    .withColumn("age_future", F.floor(F.months_between(F.col(period_col), F.col("first_sale_dt"))) + F.lit(1))
    .withColumn("r_hist_mature", F.least(F.lit(1.0), F.col("n_periods_with_sale") / F.lit(12.0)))
    .withColumn("r_hist_launch", F.least(F.lit(1.0), F.col("n_periods_with_sale") / F.lit(6.0)))
    .withColumn("r_age_launch",  F.least(F.lit(1.0), F.col("age_future") / F.lit(24.0)))
)

if USE_SQRT_PRODUCT_FOR_LAUNCH_RELIABILITY:
    base = base.withColumn("r_launch", F.sqrt(F.col("r_hist_launch") * F.col("r_age_launch")))
else:
    base = base.withColumn("r_launch", F.col("r_hist_launch") * F.col("r_age_launch"))

base = base.withColumn(
    "r_eff",
    F.when(F.col("is_launch") == 1, F.col("r_launch"))
     .otherwise(F.col("r_hist_mature"))
)

base = base.withColumn(
    "beta",
    F.when(
        F.col("is_launch") == 1,
        F.lit(BETA_LAUNCH_MIN) + (F.lit(BETA_LAUNCH_MAX) - F.lit(BETA_LAUNCH_MIN)) * (F.lit(1.0) - F.col("r_hist_launch"))
    ).otherwise(
        F.lit(BETA_MATURE_MIN) + (F.lit(BETA_MATURE_MAX) - F.lit(BETA_MATURE_MIN)) * (F.lit(1.0) - F.col("r_hist_mature"))
    )
)

base = base.withColumn(
    "micro_share_smooth",
    (F.col("micro_share") + F.col("beta") * F.col("prior_share")) / (F.lit(1.0) + F.col("beta"))
)

base = base.withColumn(
    "weight_raw",
    (F.lit(1.0) - F.col("r_eff")) * F.col("prior_share") + F.col("r_eff") * F.col("micro_share_smooth")
)

if CAP_LAUNCH is not None and CAP_MATURE is not None:
    base = (
        base
        .withColumn("cap", F.when(F.col("is_launch") == 1, F.lit(float(CAP_LAUNCH))).otherwise(F.lit(float(CAP_MATURE))))
        .withColumn("weight_cap", F.least(F.col("weight_raw"), F.col("cap")))
        .withColumn("sum_cap", F.sum("weight_cap").over(w_group))
        .withColumn(
            "weight",
            F.when(F.col("sum_cap") > 0, F.col("weight_cap") / F.col("sum_cap"))
             .otherwise(F.col("prior_share"))
        )
    )
else:
    base = base.withColumn("weight", F.col("weight_raw"))

micro_w = base.select(
    period_col, *sku_cols,
    "micro_qt_units", "prior_share", "micro_share", "micro_share_smooth",
    "r_eff", "beta", "weight"
)

#07. Allocate macro totals down to SKU

In [0]:
sku_forecast = (
    micro_w
    .join(macro, on=[period_col, *market_cols], how="left")
    .withColumn("sku_units_forecast", F.col("macro_qt_units") * F.col("weight"))
)

sku_forecast = sku_forecast.select(
    period_col, *sku_cols,
    "macro_qt_units", "micro_qt_units",
    "prior_share", "micro_share", "micro_share_smooth",
    "r_eff", "beta", "weight",
    "sku_units_forecast"
)

#08. QC: sums must match macro

In [0]:
check = (
    sku_forecast
    .groupBy(period_col, *market_cols)
    .agg(
        F.first("macro_qt_units").alias("macro_qt_units"),
        F.sum("sku_units_forecast").alias("sum_sku_forecast"),
        F.count("*").alias("n_skus")
    )
    .withColumn("diff", F.col("sum_sku_forecast") - F.col("macro_qt_units"))
)

#09. Write final table

In [0]:
df_final = (
    sku_forecast
    .select(period_col, *sku_cols, F.col("sku_units_forecast").alias(HIST_Y_COL))
    .na.fill({HIST_Y_COL: 0.0})
)

df_final.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TARGET_TABLE)